IMPORTAÇÃO DAS BIBLIOTECAS

In [1]:
# Importamos o pandas para trabalhar com tabelas de dados.
import pandas as pd
# Importamos o numpy para operações numéricas.
import numpy as np
# Importamos o módulo os para montar caminhos de arquivos de forma segura.
import os
# Importamos o matplotlib para criar gráficos estáticos.
import matplotlib.pyplot as plt
# Importamos formatadores do matplotlib para personalizar rótulos dos eixos.
import matplotlib.ticker as mticker
# Importamos o seaborn para gráficos estatísticos com estilo mais refinado.
import seaborn as sns
# Importamos o Plotly Express para construir o mapa interativo.
import plotly.express as px
# Importamos warnings para controlar mensagens de aviso durante a execução.
import warnings

MANIPULAÇÃO DA PLANILHA PARA ANÁLISE

In [2]:
# Colocar em um DataFrame para visualizar melhor e saber o número de linhas e colunas da planilha.
analise_dados_df = pd.read_csv("planilha_desproporcionalidade_02032026.csv")
analise_dados_df.columns = (
    analise_dados_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# FILTRAR PARA METILFENIDATO E LISDEXANFETAMINA
coluna_principio_ativo = "principios_ativos_whodrug" #nome da coluna que contém os princípios ativos

# Guardar o número de linhas antes do filtro
n_antes = len(analise_dados_df)

# para padronizar texto em minusculas:
analise_dados_df["principios_ativos_whodrug"] = (
    analise_dados_df["principios_ativos_whodrug"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# colocar termos de metilfenidato e lisdexanfetamina
padrao_metilfenidato = (
    "metilfenidato|"
    "methylphenidate|"
    "methylphenidate hydrochloride|"
    "cloridrato de metilfenidato"
)
padrao_lisdexanfetamina = (
    "lisdexanfetamina|"
    "lisdexanfetamine|"
    "lisdexanfetamine mesilate|"
    "mesilato de lisdexanfetamina"
)

# Criar filtros
filtro_metilfenidato = analise_dados_df[coluna_principio_ativo].str.contains(
    padrao_metilfenidato, regex=True, na=False
)
filtro_lisdexanfetamina = analise_dados_df[coluna_principio_ativo].str.contains(
    padrao_lisdexanfetamina, regex=True, na=False
)

# Nomeei uma coluna nova para identificar se o princípio ativo é metilfenidato ou lisdexanfetamina (psicoestimulante_interesse)
analise_dados_df["principio_ativo_interesse"] = "n/a"

analise_dados_df.loc[filtro_metilfenidato,
    "principio_ativo_interesse"] = "metilfenidato"

analise_dados_df.loc[filtro_lisdexanfetamina,
    "principio_ativo_interesse"] = "lisdexanfetamina"

analise_dados_df.loc[filtro_metilfenidato & filtro_lisdexanfetamina, 
    "principio_ativo_interesse"] = "metilfenidato + lisdexanfetamina" #caso raro que apareça os 2

# Filtrar a base, mantendo só linhas com metilfenidato ou lisdexanfetamina
analise_dados_df = analise_dados_df[
    filtro_metilfenidato | filtro_lisdexanfetamina
].copy()

# Conferir quantas linhas ficaram
n_depois = len(analise_dados_df)

print(f"Linhas antes do filtro: {n_antes}")
print(f"Linhas depois do filtro: {n_depois}")


# Alterei nome de colunas para facilitar analisar o que meu projeto quer.
analise_dados_df = analise_dados_df.rename(
    columns={
        "data_inicio_hora_x": "data_inicio_eam",
        "data_final_hora_x": "data_final_eam",
        "inicio_administracao": "data_inicio_uso",
        "fim_administracao": "data_fim_uso"
    }
)

# Contar quantas linhas cada notificação ocupa na base filtrada (identificacao_notificacao)
analise_dados_df["n_linhas_mesma_notificacao"] = (
    analise_dados_df
    .groupby("identificacao_notificacao")["identificacao_notificacao"]
    .transform("size"))

# Conferir
print("Número de linhas na base filtrada:", len(analise_dados_df))
print("Número de notificações únicas:", analise_dados_df["identificacao_notificacao"].nunique(dropna=True))

display(analise_dados_df.head(10))
print("Número de linhas e colunas:", analise_dados_df.shape)

C:\Users\fcfrp\AppData\Local\Temp\ipykernel_16372\2961539762.py:2: DtypeWarning: Columns (8,12,14,15,16,34,35,44,45,46,48,59) have mixed types. Specify dtype option on import or set low_memory=False.
  analise_dados_df = pd.read_csv("planilha_desproporcionalidade_02032026.csv")


Linhas antes do filtro: 1202857
Linhas depois do filtro: 3446
Número de linhas na base filtrada: 3446
Número de notificações únicas: 415


,uf,tipo_entrada_vigimed,recebido_de,identificacao_notificacao,data_inclusao_sistema,data_ultima_atualizacao,data_notificacao,tipo_notificacao,notificacao_parent_child,data_nascimento,...,posologia,duracao,data_inicio_uso,data_fim_uso,forma_farmaceutica,via_administracao,via_administracao_mae_pai,numelo_lote,principio_ativo_interesse,n_linhas_mesma_notificacao
47126,NaN,Empresas Farmacêuticas,Empresa Farmacêutica,BR-ANVISA-300041842,20210201,20210201.0,NaN,Notificação espontânea,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,desconhecida,NaN,NaN,metilfenidato,1
52604,NaN,Empresas Farmacêuticas,Empresa Farmacêutica,BR-ANVISA-300038438,20210113,20210113.0,NaN,Notificação espontânea,NaN,20120214.0,...,"30 mg, uma vez ao dia (uma cápsula) (iniciou h...",NaN,NaN,NaN,NaN,oral,NaN,NaN,metilfenidato,6
52605,NaN,Empresas Farmacêuticas,Empresa Farmacêutica,BR-ANVISA-300038438,20210113,20210113.0,NaN,Notificação espontânea,NaN,20120214.0,...,"20 mg, uma vez ao dia (uma cápsula) (iniciou h...",NaN,NaN,NaN,NaN,oral,NaN,NaN,metilfenidato,6
52606,NaN,Empresas Farmacêuticas,Empresa Farmacêutica,BR-ANVISA-300038438,20210113,20210113.0,NaN,Notificação espontânea,NaN,20120214.0,...,"10 mg, uma vez ao dia (uma cápsula) (iniciou h...",NaN,NaN,NaN,NaN,oral,NaN,NaN,metilfenidato,6
52607,NaN,Empresas Farmacêuticas,Empresa Farmacêutica,BR-ANVISA-300038438,20210113,20210113.0,NaN,Notificação espontânea,NaN,20120214.0,...,"30 mg, uma vez ao dia (uma cápsula) (iniciou h...",NaN,NaN,NaN,NaN,oral,NaN,NaN,metilfenidato,6
52608,NaN,Empresas Farmacêuticas,Empresa Farmacêutica,BR-ANVISA-300038438,20210113,20210113.0,NaN,Notificação espontânea,NaN,20120214.0,...,"20 mg, uma vez ao dia (uma cápsula) (iniciou h...",NaN,NaN,NaN,NaN,oral,NaN,NaN,metilfenidato,6
52609,NaN,Empresas Farmacêuticas,Empresa Farmacêutica,BR-ANVISA-300038438,20210113,20210113.0,NaN,Notificação espontânea,NaN,20120214.0,...,"10 mg, uma vez ao dia (uma cápsula) (iniciou h...",NaN,NaN,NaN,NaN,oral,NaN,NaN,metilfenidato,6
77030,NaN,Empresas Farmacêuticas,Empresa Farmacêutica,BR-ANVISA-300057559,20210513,20210513.0,NaN,Notificação espontânea,NaN,19981021.0,...,"2 DF, QD (pack of 30 tablets), (started more t...",NaN,NaN,NaN,MODIFIED-RELEASE TABLET,desconhecida,NaN,NaN,metilfenidato,3
77031,NaN,Empresas Farmacêuticas,Empresa Farmacêutica,BR-ANVISA-300057559,20210513,20210513.0,NaN,Notificação espontânea,NaN,19981021.0,...,"2 DF, QD (pack of 30 tablets), (started more t...",NaN,NaN,NaN,MODIFIED-RELEASE TABLET,desconhecida,NaN,NaN,metilfenidato,3
77032,NaN,Empresas Farmacêuticas,Empresa Farmacêutica,BR-ANVISA-300057559,20210513,20210513.0,NaN,Notificação espontânea,NaN,19981021.0,...,"2 DF, QD (pack of 30 tablets), (started more t...",NaN,NaN,NaN,MODIFIED-RELEASE TABLET,desconhecida,NaN,NaN,metilfenidato,3


Número de linhas e colunas: (3446, 63)


In [3]:
# Preciso priorizar a indicação terapêutica como a indicação_meddra e, caso não houver, a relatada

# Primeiro, tratar "n/a" como ausente só nessas colunas
analise_dados_df["indicacao_meddra"] = (
    analise_dados_df["indicacao_meddra"]
    .replace(["n/a", "N/A", "", "nan", "NaN"], pd.NA)
)

analise_dados_df["indicacao_relatada_notificador_inicial"] = (
    analise_dados_df["indicacao_relatada_notificador_inicial"]
    .replace(["n/a", "N/A", "", "nan", "NaN"], pd.NA)
)

# Priorizar MedDRA e, se não tiver, usa a relatada
analise_dados_df["indicacao_terapeutica"] = (
    analise_dados_df["indicacao_meddra"]
    .combine_first(analise_dados_df["indicacao_relatada_notificador_inicial"])
    .fillna("n/a")
)

# E se ainda estiver vazio, colocar "n/a"
analise_dados_df["indicacao_terapeutica"] = analise_dados_df["indicacao_terapeutica"].fillna("n/a")

# Selecionar minhas colunas de interesse.
analise_dados_interesse_df = analise_dados_df[
    ["identificacao_notificacao", "n_linhas_mesma_notificacao", "principio_ativo_interesse", "principios_ativos_whodrug", "sexo", "idade_momento_reacao", "posologia", "dose", 
     "data_inicio_uso", "indicacao_terapeutica", "data_inclusao_sistema", "notificador", "data_inicio_eam", "grave_y",
     "gravidade_y", "pt", "soc","desfecho_y"
]
]

display(analise_dados_interesse_df.head(10))
print("Número de linhas e colunas:", analise_dados_interesse_df.shape)

,identificacao_notificacao,n_linhas_mesma_notificacao,principio_ativo_interesse,principios_ativos_whodrug,sexo,idade_momento_reacao,posologia,dose,data_inicio_uso,indicacao_terapeutica,data_inclusao_sistema,notificador,data_inicio_eam,grave_y,gravidade_y,pt,soc,desfecho_y
47126,BR-ANVISA-300041842,1,metilfenidato,methylphenidate hydrochloride,Desconhecido,NaN,NaN,NaN,NaN,n/a,20210201,Consumidor ou outro não profissional de saúde,NaN,Sim,Incapacidade persistente ou significativa,Distúrbio da fala,Distúrbios do sistema nervoso,Desconhecido
52604,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"30 mg, uma vez ao dia (uma cápsula) (iniciou h...",30 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,Outro efeito clinicamente significativo,Transtorno do espectro autista,Distúrbios psiquiátricos,Desconhecido
52605,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"20 mg, uma vez ao dia (uma cápsula) (iniciou h...",20 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,Outro efeito clinicamente significativo,Transtorno do espectro autista,Distúrbios psiquiátricos,Desconhecido
52606,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"10 mg, uma vez ao dia (uma cápsula) (iniciou h...",10 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,Outro efeito clinicamente significativo,Transtorno do espectro autista,Distúrbios psiquiátricos,Desconhecido
52607,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"30 mg, uma vez ao dia (uma cápsula) (iniciou h...",30 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,"Incapacidade persistente ou significativa, Out...",Incapacidade intelectual,Distúrbios do sistema nervoso,Não Recuperado/Não Resolvido/Em andamento
52608,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"20 mg, uma vez ao dia (uma cápsula) (iniciou h...",20 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,"Incapacidade persistente ou significativa, Out...",Incapacidade intelectual,Distúrbios do sistema nervoso,Não Recuperado/Não Resolvido/Em andamento
52609,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"10 mg, uma vez ao dia (uma cápsula) (iniciou h...",10 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,"Incapacidade persistente ou significativa, Out...",Incapacidade intelectual,Distúrbios do sistema nervoso,Não Recuperado/Não Resolvido/Em andamento
77030,BR-ANVISA-300057559,3,metilfenidato,methylphenidate hydrochloride,Masculino,NaN,"2 DF, QD (pack of 30 tablets), (started more t...",2 dosage form ({DF}),NaN,Produto usado para indicação desconhecida,20210513,Consumidor ou outro não profissional de saúde,NaN,Não,NaN,Medicamento ineficaz,Distúrbios gerais e quadros clínicos no local ...,Desconhecido
77031,BR-ANVISA-300057559,3,metilfenidato,methylphenidate hydrochloride,Masculino,NaN,"2 DF, QD (pack of 30 tablets), (started more t...",2 dosage form ({DF}),NaN,Produto usado para indicação desconhecida,20210513,Consumidor ou outro não profissional de saúde,NaN,Não,NaN,Depressão,Distúrbios psiquiátricos,Desconhecido
77032,BR-ANVISA-300057559,3,metilfenidato,methylphenidate hydrochloride,Masculino,NaN,"2 DF, QD (pack of 30 tablets), (started more t...",2 dosage form ({DF}),NaN,Produto usado para indicação desconhecida,20210513,Consumidor ou outro não profissional de saúde,NaN,Sim,Ameaça à vida,Tentativa de suicídio,Distúrbios psiquiátricos,Desconhecido


Número de linhas e colunas: (3446, 18)


AJUSTE DAS DATAS

In [4]:
print(analise_dados_interesse_df["data_inclusao_sistema"].dtype)
print(analise_dados_interesse_df["data_inclusao_sistema"].head())

int64
47126    20210201
52604    20210113
52605    20210113
52606    20210113
52607    20210113
Name: data_inclusao_sistema, dtype: int64


In [5]:
coluna_data = "data_inclusao_sistema"

analise_dados_interesse_df = analise_dados_interesse_df.copy()

analise_dados_interesse_df[coluna_data] = pd.to_datetime(
    analise_dados_interesse_df[coluna_data].astype("string"), # Forcei a ler como string
    format="%Y%m%d",
    errors="coerce")

In [6]:
print(analise_dados_interesse_df[coluna_data].dtype)
print(analise_dados_interesse_df[coluna_data].head())

print(
    "Datas que não puderam ser convertidas:",
    analise_dados_interesse_df[coluna_data].isna().sum())

datetime64[ns]
47126   2021-02-01
52604   2021-01-13
52605   2021-01-13
52606   2021-01-13
52607   2021-01-13
Name: data_inclusao_sistema, dtype: datetime64[ns]
Datas que não puderam ser convertidas: 0


OUTRAS DATAS

In [7]:
for coluna in ["data_inicio_uso", "data_inicio_eam"]:

    # Converte para texto e remove espaços no início e no final
    valores_originais = (
        analise_dados_interesse_df[coluna]
        .astype("string")
        .str.strip()
    )

    # Converte valores numéricos; espaços e textos inválidos viram NaN
    valores_numericos = pd.to_numeric(
        valores_originais,
        errors="coerce"
    )

    # Retira eventual .0 e transforma novamente em texto
    valores_texto = (
        valores_numericos
        .astype("Int64")
        .astype("string")
    )

    print(f"\n{coluna}")
    print(
        valores_texto
        .str.len()
        .value_counts(dropna=False)
        .sort_index()
    )


data_inicio_uso
data_inicio_uso
4        169
6        287
8        581
<NA>    2409
Name: count, dtype: Int64

data_inicio_eam
data_inicio_eam
4        243
6        166
8        625
<NA>    2412
Name: count, dtype: Int64


In [8]:
colunas_data = ["data_inicio_uso", "data_inicio_eam"]

arquivo_saida = "datas_incompletas_inspecao.xlsx"

with pd.ExcelWriter(arquivo_saida, engine="openpyxl") as writer:

    for coluna in colunas_data:

        # Limpa espaços e converte conteúdos numéricos
        valores_originais = (
            analise_dados_interesse_df[coluna]
            .astype("string")
            .str.strip()
        )

        valores_numericos = pd.to_numeric(
            valores_originais,
            errors="coerce"
        )

        # Padroniza como texto, sem eventual ".0"
        valores_texto = (
            valores_numericos
            .astype("Int64")
            .astype("string")
        )

        numero_digitos = valores_texto.str.len()

        # Seleciona somente valores numéricos com menos de 8 dígitos
        mascara_menores = (
            valores_texto.notna()
            & numero_digitos.lt(8)
        )

        # Exporta todas as colunas das linhas correspondentes
        inspecao_df = (
            analise_dados_interesse_df
            .loc[mascara_menores]
            .copy()
        )

        # Mantém o índice da base para localizar posteriormente
        inspecao_df.insert(
            0,
            "indice_original",
            inspecao_df.index
        )

        # Acrescenta informações para facilitar a inspeção
        inspecao_df[f"{coluna}_texto"] = valores_texto.loc[mascara_menores]
        inspecao_df[f"{coluna}_numero_digitos"] = numero_digitos.loc[
            mascara_menores
        ]

        inspecao_df.to_excel(
            writer,
            sheet_name=coluna,
            index=False
        )

        print(
            f"{coluna}: {mascara_menores.sum()} valores "
            "com menos de 8 dígitos"
        )

print(f"\nArquivo exportado: {arquivo_saida}")

data_inicio_uso: 456 valores com menos de 8 dígitos
data_inicio_eam: 409 valores com menos de 8 dígitos

Arquivo exportado: datas_incompletas_inspecao.xlsx


PARA JUNTAR SINAIS DE ROR E PRR ALINHADOS NESSA PLANILHA

In [9]:
# Primeiro ler as planilhas

sinais_metilfenidato_df = pd.read_excel(
    "df_resultados_desproporcionalidade_metilfenidato_monoterapia.xlsx"
)
sinais_lisdexanfetamina_df = pd.read_excel(
    "df_resultados_desproporcionalidade_lisdexanfetamina_monoterapia.xlsx"
)

# Padronizar o nome da coluna que se chama "Evento Adverso" na planilha que criei para "evento_adverso"

sinais_metilfenidato_df.columns = (
    sinais_metilfenidato_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

sinais_lisdexanfetamina_df.columns = (
    sinais_lisdexanfetamina_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Conferir

print(sinais_metilfenidato_df.columns)
print(sinais_lisdexanfetamina_df.columns)

Index(['evento_adverso', 'a', 'c', 'b', 'd', 'ror', 'ic_lower', 'ic_upper',
       'prr', 'prr_ic_lower', 'prr_ic_upper', 'sinal_ror', 'sinal_prr', 'hlt',
       'hlgt', 'soc'],
      dtype='object')
Index(['evento_adverso', 'a', 'c', 'b', 'd', 'ror', 'ic_lower', 'ic_upper',
       'prr', 'prr_ic_lower', 'prr_ic_upper', 'sinal_ror', 'sinal_prr', 'hlt',
       'hlgt', 'soc'],
      dtype='object')


In [10]:
# Para usar o mesmo nome da base bruta
sinais_metilfenidato_df["principio_ativo_interesse"] = "metilfenidato"
sinais_lisdexanfetamina_df["principio_ativo_interesse"] = "lisdexanfetamina"

# Juntar duas planilhas de sinais
sinais_combinados_df = pd.concat(
    [sinais_metilfenidato_df, sinais_lisdexanfetamina_df],
    ignore_index=True
)

# Para preparar para o merge e renomear para pt

sinais_para_merge_df = sinais_combinados_df[
    ["principio_ativo_interesse", "evento_adverso", "sinal_ror", "sinal_prr"]
].copy()

sinais_para_merge_df = sinais_para_merge_df.rename(columns={"evento_adverso": "pt"})

# Conferir
display(sinais_para_merge_df.head())

,principio_ativo_interesse,pt,sinal_ror,sinal_prr
0,metilfenidato,medicamento ineficaz,True,True
1,metilfenidato,mal-estar,True,True
2,metilfenidato,dependência de droga ou medicamento,True,True
3,metilfenidato,ansiedade,True,True
4,metilfenidato,depressão,True,True


In [11]:
# Padronizar o PT nas duas bases para o merge funcionar
analise_dados_df["pt"] = (
    analise_dados_df["pt"]
    .astype("string")
    .str.strip()
    .str.lower()
)

sinais_para_merge_df["pt"] = (
    sinais_para_merge_df["pt"]
    .astype("string")
    .str.strip()
    .str.lower()
)


# Fazer o merge com a base de interesse, para ter os sinais de desproporcionalidade junto com os dados do EAM

analise_dados_df = analise_dados_df.merge(
    sinais_para_merge_df,
    how="left",
    on=["principio_ativo_interesse", "pt"], # Esse on quer dizer que é pra juntar quando principio_ativo_interesse e pt forem iguais
    validate="many_to_one",
    indicator="status_merge_sinal"
)

# Conferir
print(analise_dados_df["status_merge_sinal"].value_counts())

status_merge_sinal
both          3446
left_only        0
right_only       0
Name: count, dtype: int64


In [12]:
# Deixar os sinais de desproporcionalidade em verdadeiro e falso, para ficar mais fácil de ler e entender
analise_dados_df["sinal_ror"] = (
    analise_dados_df["sinal_ror"]
    .map({True: "Verdadeiro", False: "Falso"})
    .fillna("N/A")
)

analise_dados_df["sinal_prr"] = (
    analise_dados_df["sinal_prr"]
    .map({True: "Verdadeiro", False: "Falso"})
    .fillna("N/A")
)

# Agora minha planilha ficará:

analise_dados_interesse_df = analise_dados_df[
    [
        "identificacao_notificacao",
        "n_linhas_mesma_notificacao",
        "principio_ativo_interesse",
        "principios_ativos_whodrug",
        "sexo",
        "idade_momento_reacao",
        "posologia",
        "dose",
        "data_inicio_uso",
        "indicacao_terapeutica",
        "data_inclusao_sistema",
        "notificador",
        "data_inicio_eam",
        "grave_y",
        "gravidade_y",
        "pt",
        "soc",
        "desfecho_y",
        "sinal_ror",
        "sinal_prr"
    ]
].copy()

display(analise_dados_interesse_df.head(50))

,identificacao_notificacao,n_linhas_mesma_notificacao,principio_ativo_interesse,principios_ativos_whodrug,sexo,idade_momento_reacao,posologia,dose,data_inicio_uso,indicacao_terapeutica,data_inclusao_sistema,notificador,data_inicio_eam,grave_y,gravidade_y,pt,soc,desfecho_y,sinal_ror,sinal_prr
0,BR-ANVISA-300041842,1,metilfenidato,methylphenidate hydrochloride,Desconhecido,NaN,NaN,NaN,NaN,n/a,20210201,Consumidor ou outro não profissional de saúde,NaN,Sim,Incapacidade persistente ou significativa,distúrbio da fala,Distúrbios do sistema nervoso,Desconhecido,Falso,Falso
1,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"30 mg, uma vez ao dia (uma cápsula) (iniciou h...",30 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,Outro efeito clinicamente significativo,transtorno do espectro autista,Distúrbios psiquiátricos,Desconhecido,Verdadeiro,Verdadeiro
2,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"20 mg, uma vez ao dia (uma cápsula) (iniciou h...",20 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,Outro efeito clinicamente significativo,transtorno do espectro autista,Distúrbios psiquiátricos,Desconhecido,Verdadeiro,Verdadeiro
3,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"10 mg, uma vez ao dia (uma cápsula) (iniciou h...",10 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,Outro efeito clinicamente significativo,transtorno do espectro autista,Distúrbios psiquiátricos,Desconhecido,Verdadeiro,Verdadeiro
4,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"30 mg, uma vez ao dia (uma cápsula) (iniciou h...",30 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,"Incapacidade persistente ou significativa, Out...",incapacidade intelectual,Distúrbios do sistema nervoso,Não Recuperado/Não Resolvido/Em andamento,Falso,Falso
5,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"20 mg, uma vez ao dia (uma cápsula) (iniciou h...",20 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,"Incapacidade persistente ou significativa, Out...",incapacidade intelectual,Distúrbios do sistema nervoso,Não Recuperado/Não Resolvido/Em andamento,Falso,Falso
6,BR-ANVISA-300038438,6,metilfenidato,methylphenidate hydrochloride,Masculino,8 ano,"10 mg, uma vez ao dia (uma cápsula) (iniciou h...",10 milligram (mg),NaN,Transtorno de déficit de atenção e hiperatividade,20210113,Outro profissional de saúde,2019,Sim,"Incapacidade persistente ou significativa, Out...",incapacidade intelectual,Distúrbios do sistema nervoso,Não Recuperado/Não Resolvido/Em andamento,Falso,Falso
7,BR-ANVISA-300057559,3,metilfenidato,methylphenidate hydrochloride,Masculino,NaN,"2 DF, QD (pack of 30 tablets), (started more t...",2 dosage form ({DF}),NaN,Produto usado para indicação desconhecida,20210513,Consumidor ou outro não profissional de saúde,NaN,Não,NaN,medicamento ineficaz,Distúrbios gerais e quadros clínicos no local ...,Desconhecido,Verdadeiro,Verdadeiro
8,BR-ANVISA-300057559,3,metilfenidato,methylphenidate hydrochloride,Masculino,NaN,"2 DF, QD (pack of 30 tablets), (started more t...",2 dosage form ({DF}),NaN,Produto usado para indicação desconhecida,20210513,Consumidor ou outro não profissional de saúde,NaN,Não,NaN,depressão,Distúrbios psiquiátricos,Desconhecido,Verdadeiro,Verdadeiro
9,BR-ANVISA-300057559,3,metilfenidato,methylphenidate hydrochloride,Masculino,NaN,"2 DF, QD (pack of 30 tablets), (started more t...",2 dosage form ({DF}),NaN,Produto usado para indicação desconhecida,20210513,Consumidor ou outro não profissional de saúde,NaN,Sim,Ameaça à vida,tentativa de suicídio,Distúrbios psiqu

PREPARAR TOQUES FINAIS PARA EXPORTAR

In [13]:
analise_exportacao_df = analise_dados_interesse_df.copy()

# Limpar textos vazios

for coluna in analise_exportacao_df.select_dtypes(include=["object", "string"]).columns:
    analise_exportacao_df[coluna] = (
        analise_exportacao_df[coluna]
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "none": pd.NA,
            "None": pd.NA,
            "n/a": pd.NA,
            "N/A": pd.NA
        })
    )

# Transformar ausentes em N/A

analise_exportacao_df = analise_exportacao_df.fillna("N/A")

# Exportar para Excel

analise_exportacao_df.to_excel(
    "analise_descritiva_psicoestimulantes.xlsx",
    index=False)

print("Arquivo exportado com sucesso!")

Arquivo exportado com sucesso!
